### 📈 Why your Trading Strategy Sucks

##### ▶️ Related Quant Guild Videos:

- [Quant Explains Risk-Neutral Option Pricing](https://youtu.be/wYpg0TGxvgM)

- [Quant Ranks Retail Trading Mistakes that Blow Up Your Account](https://youtu.be/1mpNxBaBeOw)

- [Non-Stationarity and Why Market Timing Fails](https://youtu.be/7nvjrgqKjJE)

- [Quant Busts 3 Trading Myths with Math](https://youtu.be/wJfIk3VnubE)

- [How to Read Options Chains](https://youtu.be/RrRbz6oXwxE)

- [How to Trade the Covered Call](https://youtu.be/iPsPRQlDeTA)

###### ______________________________________________________________________________________________________________________________________

##### [🚀 Master your Quantitative Skills with Quant Guild](https://quantguild.com)

##### [⚔️ Play the Markets like an RPG](https://discourses.io)

##### [📚 Visit the Quant Guild Library for more Jupyter Notebooks](https://github.com/romanmichaelpaolucci/Quant-Guild-Library)

##### [📈 Interactive Brokers for Algorithmic Trading](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

##### [👾 Join the Quant Guild Discord Server](discord.com/invite/MJ4FU2c6c3)

---

### Why does your Trading Strategy Suck?

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# --- Parameters ---
np.random.seed(42)

n_steps = 252                 # 1 trading year
start_equity = 100.0
target_sharpe = .8
annual_vol = 0.18             # reasonable equity-curve volatility assumption
dt = 1 / 252

# Convert target Sharpe into annual drift, then daily arithmetic return params
annual_mu = target_sharpe * annual_vol
daily_mu = annual_mu / 252
daily_sigma = annual_vol / np.sqrt(252)

# Generate returns, then shift them so the realized Sharpe is close to target
raw_rets = np.random.normal(0, daily_sigma, n_steps)
realized_sigma_daily = raw_rets.std(ddof=1)
desired_mean_daily = target_sharpe * realized_sigma_daily / np.sqrt(252)
rets = raw_rets - raw_rets.mean() + desired_mean_daily

equity = start_equity * np.cumprod(1 + rets)
equity = np.insert(equity, 0, start_equity)
dates = pd.date_range(start="2025-01-01", periods=n_steps + 1, freq="B")

realized_sharpe = (rets.mean() / rets.std(ddof=1)) * np.sqrt(252)
total_return = equity[-1] / equity[0] - 1
max_drawdown = np.min(equity / np.maximum.accumulate(equity) - 1)

# --- Styling ---
off_white = "#e0e0e0"
curve_color = "#00ff88"
baseline_color = "#777777"

# --- Figure ---
fig = go.Figure()

# Baseline
fig.add_trace(go.Scatter(
    x=[dates[0], dates[-1]],
    y=[start_equity, start_equity],
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.6,
    showlegend=False,
    hoverinfo="skip"
))

# Animated equity curve
fig.add_trace(go.Scatter(
    x=[dates[0]],
    y=[equity[0]],
    mode="lines",
    line=dict(color=curve_color, width=3),
    name=f"Equity Curve · Sharpe ≈ {realized_sharpe:.2f}"
))

frames = []
slider_steps = []

for i in range(n_steps + 1):
    frame_name = f"f{i}"
    frames.append(go.Frame(
        data=[
            go.Scatter(x=[dates[0], dates[-1]], y=[start_equity, start_equity]),
            go.Scatter(x=dates[:i + 1], y=equity[:i + 1])
        ],
        name=frame_name
    ))
    slider_steps.append({
        "args": [
            [frame_name],
            {"frame": {"duration": 0, "redraw": False}, "mode": "immediate", "fromcurrent": True}
        ],
        "label": str(i),
        "method": "animate"
    })

fig.frames = frames

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

fig.update_layout(
    title=dict(
        text=(
            "Single Equity Curve Simulation"
            f"<br><sup>Realized annualized Sharpe: {realized_sharpe:.2f} · "
            f"Total return: {total_return:.1%} · Max drawdown: {max_drawdown:.1%}</sup>"
        ),
        x=0.5,
        font=dict(color=off_white)
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=650,
    width=1100,
    margin=dict(t=100, b=150, r=50, l=75),
    legend=dict(
        x=0.98,
        y=0.96,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(30,30,30,0.8)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1
    ),
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [None, {"frame": {"duration": 20, "redraw": False}, "fromcurrent": True}]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate", "fromcurrent": True}]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Step: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps
    }]
)

fig.update_xaxes(axis_style, range=[dates[0], dates[-1]], title_text="Date")
fig.update_yaxes(
    axis_style,
    range=[min(equity) * 0.96, max(equity) * 1.04],
    title_text="Equity Value"
)

fig.show()

###### ______________________________________________________________________________________________________________________________________

### These Strategies Aren't Exactly Impressive, but they are Uncorrelated

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Parameters ---
np.random.seed(42)

n_steps = 252
start_equity = 100.0
dates = pd.date_range(start="2025-01-01", periods=n_steps + 1, freq="B")

annual_vol_a = 0.18
annual_vol_b = 0.18
target_sharpe_a = .8
target_sharpe_b = .9

def generate_returns_with_target_sharpe(target_sharpe, annual_vol, n_steps):
    daily_sigma = annual_vol / np.sqrt(252)
    raw = np.random.normal(0, daily_sigma, n_steps)
    realized_sigma = raw.std(ddof=1)
    desired_mean = target_sharpe * realized_sigma / np.sqrt(252)
    return raw - raw.mean() + desired_mean

# Independent draws = uncorrelated in expectation
rets_a = generate_returns_with_target_sharpe(target_sharpe_a, annual_vol_a, n_steps)
rets_b = generate_returns_with_target_sharpe(target_sharpe_b, annual_vol_b, n_steps)

equity_a = np.insert(start_equity * np.cumprod(1 + rets_a), 0, start_equity)
equity_b = np.insert(start_equity * np.cumprod(1 + rets_b), 0, start_equity)

sharpe_a = (rets_a.mean() / rets_a.std(ddof=1)) * np.sqrt(252)
sharpe_b = (rets_b.mean() / rets_b.std(ddof=1)) * np.sqrt(252)
corr = np.corrcoef(rets_a, rets_b)[0, 1]

total_return_a = equity_a[-1] / equity_a[0] - 1
total_return_b = equity_b[-1] / equity_b[0] - 1
max_dd_a = np.min(equity_a / np.maximum.accumulate(equity_a) - 1)
max_dd_b = np.min(equity_b / np.maximum.accumulate(equity_b) - 1)

# --- Styling ---
off_white = "#e0e0e0"
strategy_a_color = "#00ff88"
strategy_b_color = "#00d1ff"
baseline_color = "#777777"

# --- Figure Setup ---
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        f"Strategy A · Sharpe ≈ {sharpe_a:.2f}",
        f"Strategy B · Sharpe ≈ {sharpe_b:.2f}"
    ),
    horizontal_spacing=0.12
)

# Base traces: order matters for frames
fig.add_trace(go.Scatter(
    x=[dates[0], dates[-1]], y=[start_equity, start_equity],
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.6, showlegend=False, hoverinfo="skip"
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=[dates[0]], y=[equity_a[0]],
    mode="lines",
    line=dict(color=strategy_a_color, width=3),
    name=f"Strategy A · Sharpe {sharpe_a:.2f}"
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=[dates[0], dates[-1]], y=[start_equity, start_equity],
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.6, showlegend=False, hoverinfo="skip"
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=[dates[0]], y=[equity_b[0]],
    mode="lines",
    line=dict(color=strategy_b_color, width=3),
    name=f"Strategy B · Sharpe {sharpe_b:.2f}"
), row=1, col=2)

# Frames
frames = []
slider_steps = []

for i in range(n_steps + 1):
    frame_name = f"f{i}"
    frames.append(go.Frame(
        data=[
            go.Scatter(x=[dates[0], dates[-1]], y=[start_equity, start_equity]),
            go.Scatter(x=dates[:i + 1], y=equity_a[:i + 1]),
            go.Scatter(x=[dates[0], dates[-1]], y=[start_equity, start_equity]),
            go.Scatter(x=dates[:i + 1], y=equity_b[:i + 1]),
        ],
        name=frame_name
    ))

    slider_steps.append({
        "args": [
            [frame_name],
            {"frame": {"duration": 0, "redraw": False}, "mode": "immediate", "fromcurrent": True}
        ],
        "label": str(i),
        "method": "animate"
    })

fig.frames = frames

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

y_min = min(equity_a.min(), equity_b.min()) * 0.96
y_max = max(equity_a.max(), equity_b.max()) * 1.04

fig.update_layout(
    title=dict(
        text=(
            "Side-by-Side Equity Curve Simulations"
            f"Realized return correlation: {corr:.2f}</sup>"
        ),
        x=0.5,
        font=dict(color=off_white)
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=650,
    width=1200,
    margin=dict(t=100, b=150, r=50, l=70),
    legend=dict(
        x=0.98,
        y=0.96,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(30,30,30,0.8)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1
    ),
    annotations=[
        dict(
            text=f"Total return: {total_return_a:.1%} Max DD: {max_dd_a:.1%}",
            x=0.23, y=1.05, xref="paper", yref="paper",
            showarrow=False, font=dict(color=off_white, size=12)
        ),
        dict(
            text=f"Total return: {total_return_b:.1%} Max DD: {max_dd_b:.1%}",
            x=0.77, y=1.05, xref="paper", yref="paper",
            showarrow=False, font=dict(color=off_white, size=12)
        )
    ],
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [None, {"frame": {"duration": 20, "redraw": False}, "fromcurrent": True}]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate", "fromcurrent": True}]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Step: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps
    }]
)

for col in [1, 2]:
    fig.update_xaxes(axis_style, range=[dates[0], dates[-1]], title_text="Date", row=1, col=col)
    fig.update_yaxes(axis_style, range=[y_min, y_max], title_text="Equity Value", row=1, col=col)

fig.show()

###### ______________________________________________________________________________________________________________________________________

### Uncorrelated Strategies Combined Yield Higher Sharpe Ratios

$$S_{\text{combined}} = \sqrt{S_A^2 + S_B^2}



In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# --- Parameters matching prior run ---
np.random.seed(42)

n_steps = 252
start_equity = 100.0
dates = pd.date_range(start="2025-01-01", periods=n_steps + 1, freq="B")

annual_vol_a = 0.18
annual_vol_b = 0.18
target_sharpe_a = 0.8
target_sharpe_b = 0.9

def generate_returns_with_target_sharpe(target_sharpe, annual_vol, n_steps):
    daily_sigma = annual_vol / np.sqrt(252)
    raw = np.random.normal(0, daily_sigma, n_steps)
    realized_sigma = raw.std(ddof=1)
    desired_mean = target_sharpe * realized_sigma / np.sqrt(252)
    return raw - raw.mean() + desired_mean

# Same independent streams as previous chart
rets_a = generate_returns_with_target_sharpe(target_sharpe_a, annual_vol_a, n_steps)
rets_b = generate_returns_with_target_sharpe(target_sharpe_b, annual_vol_b, n_steps)

# 50/50 combined strategy returns
rets_combo = 0.5 * rets_a + 0.5 * rets_b
equity_combo = np.insert(start_equity * np.cumprod(1 + rets_combo), 0, start_equity)

sharpe_a = (rets_a.mean() / rets_a.std(ddof=1)) * np.sqrt(252)
sharpe_b = (rets_b.mean() / rets_b.std(ddof=1)) * np.sqrt(252)
sharpe_combo = (rets_combo.mean() / rets_combo.std(ddof=1)) * np.sqrt(252)
corr = np.corrcoef(rets_a, rets_b)[0, 1]

total_return_combo = equity_combo[-1] / equity_combo[0] - 1
max_dd_combo = np.min(equity_combo / np.maximum.accumulate(equity_combo) - 1)

# --- Styling ---
off_white = "#e0e0e0"
combo_color = "#ffd166"
baseline_color = "#777777"

fig = go.Figure()

# Baseline
fig.add_trace(go.Scatter(
    x=[dates[0], dates[-1]],
    y=[start_equity, start_equity],
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.6,
    showlegend=False,
    hoverinfo="skip"
))

# Animated combined equity curve
fig.add_trace(go.Scatter(
    x=[dates[0]],
    y=[equity_combo[0]],
    mode="lines",
    line=dict(color=combo_color, width=3),
    name=f"50/50 Combined · Sharpe {sharpe_combo:.2f}"
))

frames = []
slider_steps = []

for i in range(n_steps + 1):
    frame_name = f"f{i}"
    frames.append(go.Frame(
        data=[
            go.Scatter(x=[dates[0], dates[-1]], y=[start_equity, start_equity]),
            go.Scatter(x=dates[:i + 1], y=equity_combo[:i + 1])
        ],
        name=frame_name
    ))

    slider_steps.append({
        "args": [
            [frame_name],
            {"frame": {"duration": 0, "redraw": False}, "mode": "immediate", "fromcurrent": True}
        ],
        "label": str(i),
        "method": "animate"
    })

fig.frames = frames

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

fig.update_layout(
    title=dict(
        text=(
            "50/50 Combined Strategy Equity Curve"
            f"<br><sup>Combined Sharpe: {sharpe_combo:.2f} · "
            f"A Sharpe: {sharpe_a:.2f} · B Sharpe: {sharpe_b:.2f} · "
            f"Correlation: {corr:.2f} · Total return: {total_return_combo:.1%} · "
            f"Max DD: {max_dd_combo:.1%}</sup>"
        ),
        x=0.5,
        font=dict(color=off_white)
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=650,
    width=1100,
    margin=dict(t=100, b=150, r=50, l=75),
    legend=dict(
        x=0.98,
        y=0.96,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(30,30,30,0.8)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1
    ),
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [None, {"frame": {"duration": 20, "redraw": False}, "fromcurrent": True}]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate", "fromcurrent": True}]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Step: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps
    }]
)

fig.update_xaxes(axis_style, range=[dates[0], dates[-1]], title_text="Date")
fig.update_yaxes(
    axis_style,
    range=[equity_combo.min() * 0.96, equity_combo.max() * 1.04],
    title_text="Equity Value"
)

fig.show()

###### ______________________________________________________________________________________________________________________________________

### It's About Physical and Stochastic Independence **NOT** Diversification

Consider these two strategies, one is based on two "uncorrelated" U.S. Equities

The other is based on two "physically independent" sources of risk: Sports Betting and Volatility Risk Premium

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Parameters ---
np.random.seed(21)

n_steps = 252
start_equity = 100.0
dates = pd.date_range(start="2025-01-01", periods=n_steps + 1, freq="B")
vol_daily = 0.14 / np.sqrt(252)

shock_start = 95
shock_end = 145

def sharpe_ratio(rets):
    return (rets.mean() / rets.std(ddof=1)) * np.sqrt(252)

def equity_curve(rets):
    return np.insert(start_equity * np.cumprod(1 + rets), 0, start_equity)

def max_drawdown(equity):
    return np.min(equity / np.maximum.accumulate(equity) - 1)

# --- Top: diversification holds ---
z_a_top = np.random.normal(0, 1, n_steps)
z_b_top = np.random.normal(0, 1, n_steps)

rets_a_top = 0.00075 + vol_daily * z_a_top
rets_b_top = 0.00070 + vol_daily * z_b_top

rets_a_top[shock_start:shock_end] += -0.0020
rets_b_top[shock_start:shock_end] += 0.00055

rets_combo_top = 0.5 * rets_a_top + 0.5 * rets_b_top

eq_a_top = equity_curve(rets_a_top)
eq_b_top = equity_curve(rets_b_top)
eq_combo_top = equity_curve(rets_combo_top)

# --- Bottom: correlation rises and both degrade ---
z_common = np.random.normal(0, 1, n_steps)
z_idio_1 = np.random.normal(0, 1, n_steps)
z_idio_2 = np.random.normal(0, 1, n_steps)

rho_normal = 0.05
rho_stress = 0.90
rho_series = np.full(n_steps, rho_normal)
rho_series[shock_start:shock_end] = rho_stress

z_a_bottom = np.sqrt(rho_series) * z_common + np.sqrt(1 - rho_series) * z_idio_1
z_b_bottom = np.sqrt(rho_series) * z_common + np.sqrt(1 - rho_series) * z_idio_2

rets_a_bottom = 0.00075 + vol_daily * z_a_bottom
rets_b_bottom = 0.00070 + vol_daily * z_b_bottom

rets_a_bottom[shock_start:shock_end] += -0.0042
rets_b_bottom[shock_start:shock_end] += -0.0037

rets_combo_bottom = 0.5 * rets_a_bottom + 0.5 * rets_b_bottom

eq_a_bottom = equity_curve(rets_a_bottom)
eq_b_bottom = equity_curve(rets_b_bottom)
eq_combo_bottom = equity_curve(rets_combo_bottom)

# Stats
top_corr = np.corrcoef(rets_a_top, rets_b_top)[0, 1]
bottom_corr_stress = np.corrcoef(rets_a_bottom[shock_start:shock_end], rets_b_bottom[shock_start:shock_end])[0, 1]
top_sharpe = sharpe_ratio(rets_combo_top)
bottom_sharpe = sharpe_ratio(rets_combo_bottom)
top_dd = max_drawdown(eq_combo_top)
bottom_dd = max_drawdown(eq_combo_bottom)

# --- Styling ---
off_white = "#e0e0e0"
combo_top_color = "#ffd166"
combo_bottom_color = "#ff6b6b"
leg_a_color = "#00ff88"
leg_b_color = "#00d1ff"
shock_color = "rgba(255, 75, 75, 0.16)"
baseline_color = "#777777"

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.13,
    subplot_titles=(
        f"Top: Diversification Holds — One Orthogonal Leg Cushions Equity Drawdown | 50/50 Sharpe: {top_sharpe:.2f}",
        f"Bottom: Diversification Fails — Correlation Rises and Both Strategies Degrade | 50/50 Sharpe: {bottom_sharpe:.2f}"
    )
)

# Top traces
fig.add_trace(go.Scatter(x=[dates[0], dates[-1]], y=[start_equity, start_equity],
                         line=dict(color=baseline_color, width=1, dash="dash"),
                         opacity=0.55, showlegend=False, hoverinfo="skip"), row=1, col=1)
fig.add_trace(go.Scatter(x=[dates[0]], y=[eq_a_top[0]], mode="lines",
                         line=dict(color=leg_a_color, width=1.5, dash="dot"),
                         name="Equity leg", opacity=0.75), row=1, col=1)
fig.add_trace(go.Scatter(x=[dates[0]], y=[eq_b_top[0]], mode="lines",
                         line=dict(color=leg_b_color, width=1.5, dash="dot"),
                         name="Orthogonal leg", opacity=0.75), row=1, col=1)
fig.add_trace(go.Scatter(x=[dates[0]], y=[eq_combo_top[0]], mode="lines",
                         line=dict(color=combo_top_color, width=3),
                         name="50/50 combined · diversification holds"), row=1, col=1)

# Bottom traces
fig.add_trace(go.Scatter(x=[dates[0], dates[-1]], y=[start_equity, start_equity],
                         line=dict(color=baseline_color, width=1, dash="dash"),
                         opacity=0.55, showlegend=False, hoverinfo="skip"), row=2, col=1)
fig.add_trace(go.Scatter(x=[dates[0]], y=[eq_a_bottom[0]], mode="lines",
                         line=dict(color=leg_a_color, width=1.5, dash="dot"),
                         name="Equity leg under stress", opacity=0.75, showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(x=[dates[0]], y=[eq_b_bottom[0]], mode="lines",
                         line=dict(color=leg_b_color, width=1.5, dash="dot"),
                         name="Second leg under stress", opacity=0.75, showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(x=[dates[0]], y=[eq_combo_bottom[0]], mode="lines",
                         line=dict(color=combo_bottom_color, width=3),
                         name="50/50 combined · correlation breakdown"), row=2, col=1)

for row in [1, 2]:
    fig.add_vrect(
        x0=dates[shock_start],
        x1=dates[shock_end],
        fillcolor=shock_color,
        line_width=0,
        layer="below",
        row=row,
        col=1
    )

frames = []
slider_steps = []

for i in range(n_steps + 1):
    frame_name = f"f{i}"
    frames.append(go.Frame(
        data=[
            go.Scatter(x=[dates[0], dates[-1]], y=[start_equity, start_equity]),
            go.Scatter(x=dates[:i + 1], y=eq_a_top[:i + 1]),
            go.Scatter(x=dates[:i + 1], y=eq_b_top[:i + 1]),
            go.Scatter(x=dates[:i + 1], y=eq_combo_top[:i + 1]),
            go.Scatter(x=[dates[0], dates[-1]], y=[start_equity, start_equity]),
            go.Scatter(x=dates[:i + 1], y=eq_a_bottom[:i + 1]),
            go.Scatter(x=dates[:i + 1], y=eq_b_bottom[:i + 1]),
            go.Scatter(x=dates[:i + 1], y=eq_combo_bottom[:i + 1]),
        ],
        name=frame_name
    ))

    slider_steps.append({
        "args": [[frame_name], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate", "fromcurrent": True}],
        "label": str(i),
        "method": "animate"
    })

fig.frames = frames

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

all_equity = np.concatenate([eq_a_top, eq_b_top, eq_combo_top, eq_a_bottom, eq_b_bottom, eq_combo_bottom])
y_min = all_equity.min() * 0.96
y_max = all_equity.max() * 1.04

fig.update_layout(
    title=dict(
        text=(
            "50/50 Strategy Under Drawdown: Diversification vs Correlation Breakdown"
            f"<br><sup>Top full-period corr: {top_corr:.2f}, max DD: {top_dd:.1%} · "
            f"Bottom stress-window corr: {bottom_corr_stress:.2f}, max DD: {bottom_dd:.1%}</sup>"
        ),
        x=0.5,
        font=dict(color=off_white)
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=850,
    width=1200,
    margin=dict(t=115, b=150, r=55, l=75),
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {"label": "▶ Play", "method": "animate",
             "args": [None, {"frame": {"duration": 20, "redraw": False}, "fromcurrent": True}]},
            {"label": "⏸ Pause", "method": "animate",
             "args": [[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate", "fromcurrent": True}]}
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {"font": {"size": 14, "color": off_white}, "prefix": "Step: ", "visible": True, "xanchor": "right"},
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps
    }],
    annotations=[
        dict(text="Equity-leg drawdown<br>orthogonal leg keeps working",
             x=dates[(shock_start + shock_end) // 2], y=y_max * 0.99,
             xref="x", yref="y", showarrow=False,
             font=dict(color=off_white, size=12),
             bgcolor="rgba(30,30,30,0.65)", bordercolor="rgba(255,255,255,0.25)"),
        dict(text="Stress regime:<br>correlation rises + both legs degrade",
             x=dates[(shock_start + shock_end) // 2], y=y_max * 0.99,
             xref="x2", yref="y2", showarrow=False,
             font=dict(color=off_white, size=12),
             bgcolor="rgba(30,30,30,0.65)", bordercolor="rgba(255,255,255,0.25)")
    ]
)

fig.update_xaxes(axis_style, range=[dates[0], dates[-1]], title_text="", row=1, col=1)
fig.update_xaxes(axis_style, range=[dates[0], dates[-1]], title_text="Date", row=2, col=1)
fig.update_yaxes(axis_style, range=[y_min, y_max], title_text="Equity Value", row=1, col=1)
fig.update_yaxes(axis_style, range=[y_min, y_max], title_text="Equity Value", row=2, col=1)

fig.show()

---

#### 💭 Closing Thoughts and Future Topics

 **📑 TL;DW Executive Summary**

  - There is no single trading strategy that exists to print you money
  - Sharpe ratios are roughly additive in the squared-distance sense when strategies are orthogonal
  - Correlation structures are arbitrary, there is no *structure* gaurenteeing them and during crisis they decompose
  - Physical independence always implies stochastic independence which enables better positioning and survival

 
**Future Topics**

Technical Videos and Other Discussions

 - Fama-French / Carhart and Factor Modeling in General
 - Hawkes Processes
 - Merton Jump Diffusion Model (and Characteristic Function Pricing, Carr-Madan 1999)
 - Market-Making Models and Simulation (Stoikov-Avellaneda)
 - My First Year as a Quant
 - Why Hedge Funds are Actually Secretive
 - Non-Markovian Models (fractional Brownian motion, Volterra Process)
 - Top 3 Uses of Linear Algebra for Quant Finance
 - Girsanov's Change of Measure
 - Rough Path Theory, Applications of Path Signatures
 - Sig-Vol Model, Calibration, and Pricing
 - Trading with Alternative Data Sources
 - Pairs Trading and Statistical Arbitrage
 - Data Cleaning & Outlier Handling in Financial Time Series
 - Practical Issues in Multi-Asset Portfolio Backtesting
 - Risk Premia Harvesting: Equity, FX, Rates

[Ideas for Interactive Brokers Apps and Tutorials](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

- How Interactive Broker's API Works (EWrapper/EClient)
- How to Backtest a Trading Strategy with Interactive Brokers
- Algorithmic Volatility Trading System

---

####  $\text{Copyright © 2026 Quant Guild} \quad \quad \quad \quad \text{Author: Roman Paolucci}$